In [2]:
import gymnasium as gym
import numpy as np
import math
import os
import configparser
from sb3_contrib.ppo_mask import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
from sb3_contrib.common.maskable.utils import get_action_masks



In [3]:
from src.hpc_env import HPCenv
from src.validation import Validation
from src.baseline import PercentileBaseline
from src.utils import mask_fn, get_config_as_dict
from src.carbon_intensity import CarbonIntensity

In [4]:

# Load config with explicit path and typed parsing
config = configparser.ConfigParser()
config_path = os.path.join(os.getcwd(), 'config_file', 'config.ini')
config.read(config_path)

['/Users/mikkeldahl/green-hpc-scheduler/config_file/config.ini']

## Model validation

In [8]:
val = Validation()
val.load_dir("results/CI_B16384_RC_LR-00030_ETA0.0001_C-None_Lu__20251016-204316Z-338b21")
processed_stats_baselines, raw_stats_baselines = val.run_baselines(n_eval_episodes=1, mode="validation")

Executing baseline:  FCFS Baseline
Episode  0
Current secounds after start:  3520255
3482
Current secounds after start:  6602707
6891
Executing baseline:  10-percentile Baseline
Episode  0
Current secounds after start:  2180555
1230
Current secounds after start:  4266462
2490
Current secounds after start:  6687519
2837
Current secounds after start:  9452309
2890
Current secounds after start:  12126713
4421
Current secounds after start:  15142354
4737
Current secounds after start:  17835172
5701
Current secounds after start:  20872559
6491
Current secounds after start:  23850156
6653
Current secounds after start:  26655204
7858
Current secounds after start:  29716180
8139
Current secounds after start:  32777687
8163
Executing baseline:  25-percentile Baseline
Episode  0
Current secounds after start:  2120561
1641
Current secounds after start:  4138573
3551
Current secounds after start:  6471978
4776
Current secounds after start:  8983558
5560
Current secounds after start:  11448316
8054

In [9]:
processed_stats_baselines

{'FCFS Baseline': {'Validation Reward': -31170.86584830056,
  'val_objective': np.float64(1.5105698251770363),
  'Avg Wait': 6241.402740396379,
  'Max Wait': 533605.0,
  'Avg Response': 10793.021531685834,
  'Avg Slowdown': 177.4515107448421,
  'Episode Duration': 7837750.0,
  'Carbon Emissions': 28091754.682081796,
  'Weighted Carbon Emissions': -28091754.682081796,
  'Reward Wait Component': -0.025508613000000013,
  'Reward Carbon Component': -31170.84033968731,
  'Reward Total Component': -31170.86584830056,
  'System Utilization': 0.5823901774464929,
  'Action Analysis': {'Total Actions': 23851,
   'Schedule Action Percentage': 34.271099744245525,
   'Fixed Delay Percentage': 0.1383589786591757,
   'Wait Delay Percentage': 65.5905412770953,
   'Fixed Delays': {'300s': 33},
   'Wait for Jobs': {'1 jobs': 15644}}},
 '10-percentile Baseline': {'Validation Reward': -22519.253874803842,
  'val_objective': np.float64(1.2672950075257094),
  'Avg Wait': 8541616.890017128,
  'Max Wait': 255

In [ ]:
val = Validation()
val.load_dir("results/CI_B8192_RC_LR-00001_ETA1.0_C-None_Lu__20251012-163316Z-d4e4b0")
processed_stats_baselines, raw_stats_baselines = val.run_baselines(n_eval_episodes=1, mode="test")

## Reconstructing carbon emissions

In [5]:
train_and_eval_folder = "results/train_and_eval/eta_1_sparse_rewards/user_ci_disabled/eta_1"

In [7]:
seeds = 5
results = {}

for seed in range(1,seeds+1):
    results[seed] = {}
    results[seed]["processed_results"] = {}
    results[seed]["trace"] = {}
for seed in range(1,seeds+1):
    val = Validation()
    val.load_dir(train_and_eval_folder)
    model_loc = train_and_eval_folder + "/seed_" + str(seed) + "/" "best_model"
    model = MaskablePPO.load(path=model_loc)
    processed_results, _ = val.validate_model(1,model,"test")
    results[seed]["processed_results"] = processed_results


{'use_constant_power': True, 'constant_power_per_processor': 500, 'procs_per_node': 1, 'idle_power': 15, 'carbon_year': 2021, 'custom_intensity': 'False ## If true it utilizes the a custom intensity, else it use real data', 'green_forecast_length': 24, 'max_queue_size': 256, 'run_win_length': 64, 'delay_time_list': [300, 600, 1200, 1800, 2400, 3600, 7200, 14400], 'max_wait_n_jobs': 4, 'job_feature': 5, 'run_feature': 2, 'green_feature_pr_timeslot': 1, 'green_feature_constant': 8, 'episode_length': 64, 'gamma': 0.99, 'gae_lambda': 0.97, 'batch_size': 8192, 'seed': 1, 'n_epochs': 2, 'pi_nn': [4000, 1000], 'vf_nn': [1024, 512], 'n_steps': 65536, 'total_timesteps': 3000000, 'ent_coef': 0, 'learning_rate': 0.0001, 'clip_range': 0.1, 'vf_coef': 0.5, 'clip_range_vf': 0.1, 'normalize_advantage': True, 'max_grad_norm': 0.5, 'n_envs': 8, 'sweep_seeds': [1, 2, 3], 'validation_freq': 65536, 'validation_episodes': 1, 'base_line_wait_carbon_penality': 0.01, 'eta': 1, 'reward_type': 'wait_abs_ems_cli

In [1]:
ce = []
for seed in range(1,seeds+1):
    print("Seed: ", seed)
    ce.append(results[seed]["model"]["Carbon Emissions"])

np.mean(ce)

NameError: name 'seeds' is not defined